In [44]:
import pandas as p
import datetime

In [45]:
customers = p.read_csv(r'Data\03_Library SystemCustomers.csv')
checkouts = p.read_csv(r'Data\03_Library Systembook.csv')

In [46]:
print(customers.to_string())

   Customer ID     Customer Name
0          1.0          Jane Doe
1          2.0        John Smith
2          3.0        Dan Reeves
3          NaN               NaN
4          5.0    William Holden
5          6.0     Jaztyn Forest
6          7.0     Jackie Irving
7          8.0  Matthew Stirling
8          9.0         Emory Ted


In [47]:
print(checkouts.to_string())

       Id                                     Books Book checkout Book Returned Days allowed to borrow  Customer ID
0     1.0                       Catcher in the Rye   "20/02/2023"    25/02/2023                2 weeks          1.0
1     2.0          Lord of the rings the two towers  "24/03/2023"    21/03/2023                2 weeks          2.0
2     3.0  Lord of the rings the return of the kind  "29/03/2023"    25/03/2023                2 weeks          3.0
3     4.0                                The hobbit  "02/04/2023"    25/03/2023                2 weeks          4.0
4     5.0                                     Dune   "02/04/2023"    25/03/2023                2 weeks          5.0
5     6.0                              Little Women  "02/04/2023"    01/05/2023                2 weeks          1.0
6     7.0                                        IT  "10/04/2063"    03/04/2023                2 weeks          6.0
7     8.0                                   Misery   "15/04/2023"    03/

In [48]:
# get initial row count for each file
initial_rows_customers = len(customers)
initial_row_checkouts = len(checkouts)

print(f"Initial #rows in Customers: {initial_rows_customers}")
print(f"Initial #rows in Checkouts: {initial_row_checkouts}")

Initial #rows in Customers: 9
Initial #rows in Checkouts: 114


In [49]:
#copy from customers excluding any nulls
customers_clean = customers.dropna().copy()

# strip whitespaces 
customers_clean['Customer Name'] = customers_clean['Customer Name'].astype(str).str.strip()

#convert ID to integer
customers_clean['Customer ID'] = customers_clean['Customer ID'].astype(int)

In [50]:
print(customers_clean.to_string())

   Customer ID     Customer Name
0            1          Jane Doe
1            2        John Smith
2            3        Dan Reeves
4            5    William Holden
5            6     Jaztyn Forest
6            7     Jackie Irving
7            8  Matthew Stirling
8            9         Emory Ted


In [ ]:
# copy from checkouts excluding nulls
checkouts_clean = checkouts.dropna().copy()

# convert itds to integer as done with customers
checkouts_clean['Id'] = checkouts_clean['Id'].astype(int)
checkouts_clean['Customer ID'] = checkouts_clean['Customer ID'].astype(int)

#fix formatting issues 
checkouts_clean['Books'] = checkouts_clean['Books'].astype(str).str.strip()
checkouts_clean['Book checkout'] = checkouts_clean['Book checkout'].astype(str).str.strip().str.replace(r'["\']', '', regex=True).str.strip()
checkouts_clean['Book Returned'] = checkouts_clean['Book Returned'].astype(str).str.strip().str.replace(r'["\']', '', regex=True).str.strip()

#converting dates
checkouts_clean['Book Checkout Date'] = p.to_datetime(checkouts_clean['Book checkout'], format='%d/%m/%Y', errors='coerce')
checkouts_clean['Book Returned Date'] = p.to_datetime(checkouts_clean['Book Returned'], format='%d/%m/%Y', errors='coerce')

#Corrupt dates
checkouts_clean = checkouts_clean.dropna(subset=['Book Checkout Date','Book Returned Date'])

#Future Dates
checkouts_clean = checkouts_clean[checkouts_clean['Book Checkout Date'].dt.year <= datetime.datetime.now().year]

# chronological issues 
checkouts_clean = checkouts_clean[checkouts_clean['Book Returned Date'] >= checkouts_clean['Book Checkout Date']]

#duplicates
checkouts_clean = checkouts_clean.drop_duplicates(subset=['Books', 'Book checkout', 'Customer ID'])

#drop the original date columns
final_checkouts = checkouts_clean.drop(columns=['Book checkout', 'Book Returned'])



In [67]:
print(final_checkouts.to_string())

    Id                    Books Days allowed to borrow  Customer ID Book Checkout Date Book Returned Date
0    1       Catcher in the Rye                2 weeks            1         2023-02-20         2023-02-25
5    6             Little Women                2 weeks            1         2023-04-02         2023-05-01
8    9                 Catch 22                2 weeks            7         2023-04-15         2023-04-16
9   10              Animal Farm                2 weeks            2         2023-04-20         2023-04-24
10  11                     1984                2 weeks            8         2023-04-23         2023-04-27
11  12             Little Women                2 weeks            1         2023-04-02         2023-05-01
12  13             East of Eden                2 weeks            2         2023-04-30         2023-05-05
13  14  America Is in the Heart                2 weeks            3         2023-05-01         2023-05-07
14  15        Wuthering Heights               

In [71]:
final_rows_checkout = len(final_checkouts)
dropped_rows_checkout = initial_row_checkouts - final_rows_checkout
print(f"Total rows dropped from checkouts: {dropped_rows_checkout}")

final_rows_customers = len(customers_clean)
dropped_rows_customers = initial_rows_customers - final_rows_customers
print(f"Total rows dropped from customers: {dropped_rows_customers}")

Total rows dropped from checkouts: 101
Total rows dropped from customers: 1


In [74]:
#exporting cleaned data

final_checkouts.to_csv('Data/Clean_CheckOuts.csv')
customers_clean.to_csv('Data/Clean_Customers.csv')